# 第 2 章 练习题答案

> 对应官方 `ch02/01_main-chapter-code/exercise-solutions.ipynb`，精选 2 道核心练习。

> 💡 建议先自己尝试，再对照答案。

## 练习 2.1：BPE 编码探究

**题目**：用 tiktoken 编码一个生造词（字典里没有的词），观察 BPE 如何处理它。

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

# 一个生造词（GPT-2 训练时没见过的组合）
word = "Akwirw ier"
ids = tokenizer.encode(word)
print(f"生造词 {word!r} → token ids: {ids}")
print(f"共 {len(ids)} 个 token，逐个解码:")
for i in ids:
    print(f"  {i} → {tokenizer.decode([i])!r}")
print("\n💡 BPE 把未知词拆成已知子词（Ak|w|ir|w| |ier），")
print("   词表固定（50257），遇到任何新词都能处理，不会映射成 <unk>。")

## 练习 2.2：数据加载器参数的影响

**题目**：对比不同 `stride` 对生成的样本数有什么影响？为什么训练时通常用 `stride = max_length`？

In [ ]:
from pathlib import Path
from src.gpt import create_dataloader_v1

data_path = Path("data/the-verdict.txt")
if not data_path.exists():
    data_path = Path("../data/the-verdict.txt")
text = data_path.read_text(encoding="utf-8")

print(f"语料: {len(text)} 字符\n")
print(f"{'stride':<8} {'max_len':<10} {'样本数':<10} {'批次(batch=4)':<15}")
print("-" * 45)
for stride in [1, 2, 4]:
    dl = create_dataloader_v1(text, batch_size=4, max_length=4,
                              stride=stride, shuffle=False, drop_last=False)
    print(f"{stride:<8} {4:<10} {len(dl.dataset):<10} {len(dl):<15}")
print("\n💡 stride=1 时样本高度重叠（数据膨胀 4 倍，信息冗余）；")
print("   stride=max_length 时无重叠，样本数最少但无冗余 → 训练用这个。")